<a href="https://colab.research.google.com/github/saikrishnanallavula/Final-year-project-Sai-Krishna-Nallavula/blob/main/Model_Training_TB_hypertuning_done_(2)_(4).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tawsifurrahman/tuberculosis-tb-chest-xray-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'tuberculosis-tb-chest-xray-dataset' dataset.
Path to dataset files: /kaggle/input/tuberculosis-tb-chest-xray-dataset


In [ ]:
!pip install -q keras-tuner

In [ ]:
import os

In [ ]:
os.listdir("/kaggle/input/tuberculosis-tb-chest-xray-dataset")

['TB_Chest_Radiography_Database']

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from tqdm import tqdm

from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications import MobileNetV2, VGG16
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout,
    GlobalAveragePooling2D, Input
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import keras_tuner as kt


# =========================================================
# 1. LOAD DATA
# =========================================================
path = Path('/kaggle/input/tuberculosis-tb-chest-xray-dataset/TB_Chest_Radiography_Database')

label_map = {
    'Normal': 0,
    'Tuberculosis': 1
}

class_names = ['Normal', 'Tuberculosis']

images = []
labels = []

for class_name, label in label_map.items():
    class_dir = path / class_name
    for img_path in tqdm(list(class_dir.iterdir()), desc=f"Loading {class_name}"):
        img = load_img(img_path, target_size=(224, 224))
        img = img_to_array(img)
        images.append(img)
        labels.append(label)

images = np.array(images, dtype='float32') / 255.0
labels = np.array(labels, dtype='int32')

# Keep integer labels for metrics
labels_int = labels.copy()

# One-hot labels for training
labels_cat = to_categorical(labels, num_classes=2)

X_train, X_val, y_train, y_val, y_train_int, y_val_int = train_test_split(
    images,
    labels_cat,
    labels_int,
    test_size=0.2,
    random_state=42,
    stratify=labels_int
)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)


# =========================================================
# 2. COMMON CALLBACKS
# =========================================================
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)


# =========================================================
# 3. MODEL BUILDERS FOR HYPERPARAMETER TUNING
# =========================================================
def build_cnn_model(hp):
    model = Sequential()

    model.add(Input(shape=(224, 224, 3)))

    # Conv Block 1
    model.add(
        Conv2D(
            filters=hp.Choice('cnn_filters_1', [32, 64]),
            kernel_size=(3, 3),
            activation='relu'
        )
    )
    model.add(MaxPooling2D(pool_size=(2, 2)))

    # Conv Block 2
    model.add(
        Conv2D(
            filters=hp.Choice('cnn_filters_2', [64, 128]),
            kernel_size=(3, 3),
            activation='relu'
        )
    )
    model.add(MaxPooling2D(pool_size=(2, 2)))

    # Conv Block 3
    model.add(
        Conv2D(
            filters=hp.Choice('cnn_filters_3', [128, 256]),
            kernel_size=(3, 3),
            activation='relu'
        )
    )
    model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Flatten())

    model.add(
        Dense(
            units=hp.Choice('cnn_dense_units', [128, 256, 512]),
            activation='relu'
        )
    )
    model.add(
        Dropout(
            rate=hp.Choice('cnn_dropout', [0.3, 0.4, 0.5])
        )
    )
    model.add(Dense(2, activation='softmax'))

    lr = hp.Choice('cnn_learning_rate', [1e-2, 1e-3, 1e-4])

    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


def build_mobilenet_model(hp):
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )

    # Tune how many layers to unfreeze
    unfreeze_layers = hp.Choice('mobilenet_unfreeze_layers', [0, 20, 50])

    if unfreeze_layers == 0:
        for layer in base_model.layers:
            layer.trainable = False
    else:
        for layer in base_model.layers[:-unfreeze_layers]:
            layer.trainable = False
        for layer in base_model.layers[-unfreeze_layers:]:
            layer.trainable = True

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(
        units=hp.Choice('mobilenet_dense_units', [128, 256, 512]),
        activation='relu'
    )(x)
    x = Dropout(
        rate=hp.Choice('mobilenet_dropout', [0.3, 0.4, 0.5])
    )(x)
    output = Dense(2, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=output)

    lr = hp.Choice('mobilenet_learning_rate', [1e-3, 1e-4, 1e-5])

    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


def build_vgg16_model(hp):
    base_model = VGG16(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )

    unfreeze_layers = hp.Choice('vgg16_unfreeze_layers', [0, 4, 8])

    if unfreeze_layers == 0:
        for layer in base_model.layers:
            layer.trainable = False
    else:
        for layer in base_model.layers[:-unfreeze_layers]:
            layer.trainable = False
        for layer in base_model.layers[-unfreeze_layers:]:
            layer.trainable = True

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(
        units=hp.Choice('vgg16_dense_units', [128, 256, 512]),
        activation='relu'
    )(x)
    x = Dropout(
        rate=hp.Choice('vgg16_dropout', [0.3, 0.4, 0.5])
    )(x)
    output = Dense(2, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=output)

    lr = hp.Choice('vgg16_learning_rate', [1e-3, 1e-4, 1e-5])

    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


# =========================================================
# 4. TUNE EACH MODEL
# =========================================================
def tune_model(build_fn, project_name):
    tuner = kt.Hyperband(
        build_fn,
        objective='val_accuracy',
        max_epochs=10,
        factor=3,
        directory='kt_dir',
        project_name=project_name,
        overwrite=True
    )

    tuner.search(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=10,
        batch_size=32,
        callbacks=[early_stop],
        verbose=1
    )

    best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
    best_model = tuner.get_best_models(num_models=1)[0]

    return tuner, best_hp, best_model


print("\nTuning CNN...")
cnn_tuner, best_hp_cnn, best_cnn_model = tune_model(build_cnn_model, 'cnn_tuning')

print("\nTuning MobileNetV2...")
mobilenet_tuner, best_hp_mobilenet, best_mobilenet_model = tune_model(build_mobilenet_model, 'mobilenet_tuning')

print("\nTuning VGG16...")
vgg16_tuner, best_hp_vgg16, best_vgg16_model = tune_model(build_vgg16_model, 'vgg16_tuning')


# =========================================================
# 5. RETRAIN BEST MODELS
# =========================================================
def train_best_model(model, model_name):
    print(f"\nTraining best {model_name} model...")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=10,
        batch_size=32,
        callbacks=[early_stop],
        verbose=1
    )
    return history


history_cnn = train_best_model(best_cnn_model, "CNN")
history_mobilenet = train_best_model(best_mobilenet_model, "MobileNetV2")
history_vgg16 = train_best_model(best_vgg16_model, "VGG16")


# =========================================================
# 6. PLOT TRAINING HISTORY
# =========================================================
def plot_history(history, model_name):
    plt.figure(figsize=(8, 5))
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f'{model_name} Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'{model_name} Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()


plot_history(history_cnn, 'CNN')
plot_history(history_mobilenet, 'MobileNetV2')
plot_history(history_vgg16, 'VGG16')


# =========================================================
# 7. EVALUATION HELPERS
# =========================================================
def evaluate_model(model, X_val, y_val_int, model_name):
    y_prob = model.predict(X_val, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    acc = accuracy_score(y_val_int, y_pred)
    prec = precision_score(y_val_int, y_pred, average='binary', zero_division=0)
    rec = recall_score(y_val_int, y_pred, average='binary', zero_division=0)
    f1 = f1_score(y_val_int, y_pred, average='binary', zero_division=0)

    print(f"\n{model_name} Classification Report:")
    print(classification_report(y_val_int, y_pred, target_names=class_names, zero_division=0))

    return {
        'model_name': model_name,
        'y_pred': y_pred,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1_score': f1
    }


cnn_results = evaluate_model(best_cnn_model, X_val, y_val_int, 'CNN')
mobilenet_results = evaluate_model(best_mobilenet_model, X_val, y_val_int, 'MobileNetV2')
vgg16_results = evaluate_model(best_vgg16_model, X_val, y_val_int, 'VGG16')


# =========================================================
# 8. CONFUSION MATRIX WITH SEABORN
# =========================================================
def plot_confusion_matrix(y_true, y_pred, model_name, class_names):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.title(f'{model_name} Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()


plot_confusion_matrix(y_val_int, cnn_results['y_pred'], 'CNN', class_names)
plot_confusion_matrix(y_val_int, mobilenet_results['y_pred'], 'MobileNetV2', class_names)
plot_confusion_matrix(y_val_int, vgg16_results['y_pred'], 'VGG16', class_names)


# =========================================================
# 9. SEPARATE BAR GRAPHS FOR ACCURACY, PRECISION, RECALL, F1
# =========================================================
all_results = [cnn_results, mobilenet_results, vgg16_results]
model_names = [r['model_name'] for r in all_results]

accuracies = [r['accuracy'] for r in all_results]
precisions = [r['precision'] for r in all_results]
recalls = [r['recall'] for r in all_results]
f1_scores = [r['f1_score'] for r in all_results]


def plot_metric_bar(metric_values, metric_name):
    plt.figure(figsize=(8, 5))
    sns.barplot(x=model_names, y=metric_values, palette='viridis')
    plt.title(f'{metric_name} Comparison')
    plt.xlabel('Model')
    plt.ylabel(metric_name)
    plt.ylim(0, 1)

    for i, v in enumerate(metric_values):
        plt.text(i, v + 0.01, f'{v:.4f}', ha='center')

    plt.show()


plot_metric_bar(accuracies, 'Accuracy')
plot_metric_bar(precisions, 'Precision')
plot_metric_bar(recalls, 'Recall')
plot_metric_bar(f1_scores, 'F1 Score')


# =========================================================
# 10. PRINT BEST HYPERPARAMETERS
# =========================================================
print("\nBest CNN Hyperparameters:")
for key, value in best_hp_cnn.values.items():
    print(f"{key}: {value}")

print("\nBest MobileNetV2 Hyperparameters:")
for key, value in best_hp_mobilenet.values.items():
    print(f"{key}: {value}")

print("\nBest VGG16 Hyperparameters:")
for key, value in best_hp_vgg16.values.items():
    print(f"{key}: {value}")

Loading Tuberculosis: 100%|██████████| 700/700 [00:08<00:00, 86.07it/s]
